In [22]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from pathlib import Path
from PIL import Image
import urllib.request
import json

IMAGES_DIR = Path(r"C:\Users\53002\Desktop\birds")
TOP_K = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {DEVICE}")

使用设备: cpu


In [24]:
# 加载预训练模型（权重自动从 PyTorch 官方镜像下载，通常国内可通）
model = models.resnet50(pretrained=True)
model = model.to(DEVICE)
model.eval()
print("模型加载完成（ImageNet 预训练）")

C:\Users\53002\anaconda3\envs\whisper_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\53002\anaconda3\envs\whisper_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to ./torch_cache\hub\checkpoints\resnet50-0676ba61.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:03<00:00, 28.7MB/s]


模型加载完成（ImageNet 预训练）


In [26]:
CLASSES_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
CLASSES_PATH = Path("imagenet_classes.txt")

if not CLASSES_PATH.exists():
    print("正在下载 ImageNet 类别标签...")
    # 使用镜像加速（如果直连慢）
    try:
        urllib.request.urlretrieve(CLASSES_URL, CLASSES_PATH)
    except:
        mirror_url = "https://ghproxy.com/" + CLASSES_URL
        urllib.request.urlretrieve(mirror_url, CLASSES_PATH)
    print("下载完成")

with open(CLASSES_PATH, 'r') as f:
    class_names = [line.strip() for line in f.readlines()]
print(f"已加载 {len(class_names)} 个类别，其中鸟类约占 60 种")

已加载 1000 个类别，其中鸟类约占 60 种


In [28]:
def preprocess_image(image_path: Path) -> torch.Tensor:
    img = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return transform(img).unsqueeze(0).to(DEVICE)

In [30]:
def predict_bird(img_tensor, top_k=TOP_K, only_bird=True):
    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
        # 获取所有类别概率
        probs_np = probs.cpu().numpy()
        # 如果只显示鸟类，可以过滤包含 'bird' 或具体鸟名的类别
        # 为了简单，我们先输出所有类别的前 top_k，然后人工识别是否为鸟
        top_probs, top_indices = torch.topk(probs, top_k * 2)  # 多取一些备用
        
    results = []
    for prob, idx in zip(top_probs.cpu(), top_indices.cpu()):
        label = class_names[idx]
        # 简单过滤：如果只想看鸟类，检查标签中是否包含 "bird" 或者常见鸟名关键词
        # 这里不过滤，直接返回，用户可自行判断
        results.append((label, prob.item()))
    return results[:top_k]

In [32]:
if not IMAGES_DIR.exists():
    print(f"错误：目录 {IMAGES_DIR} 不存在")
else:
    image_paths = sorted(IMAGES_DIR.glob("*.jpg"))
    if not image_paths:
        print(f"在 {IMAGES_DIR} 中没有找到 .jpg 文件")
    else:
        print(f"找到 {len(image_paths)} 张图片：{', '.join(p.name for p in image_paths)}\n")
        for img_path in image_paths:
            print(f"正在识别: {img_path.name}")
            try:
                img_tensor = preprocess_image(img_path)
                predictions = predict_bird(img_tensor)
                print(f"  Top {TOP_K} 预测结果:")
                for rank, (name, conf) in enumerate(predictions, start=1):
                    print(f"    {rank}. {name} (置信度: {conf:.3f})")
                print()
            except Exception as e:
                print(f"  识别失败: {e}\n")

找到 6 张图片：001.jpg, 002.jpg, 003.jpg, 004.jpg, 005.jpg, af640d1b934568d3cd61a003444d0a2e.jpg

正在识别: 001.jpg
  Top 3 预测结果:
    1. spoonbill (置信度: 0.483)
    2. flamingo (置信度: 0.365)
    3. goldfish (置信度: 0.043)

正在识别: 002.jpg
  Top 3 预测结果:
    1. goose (置信度: 0.362)
    2. black stork (置信度: 0.225)
    3. kite (置信度: 0.105)

正在识别: 003.jpg
  Top 3 预测结果:
    1. kite (置信度: 0.744)
    2. vulture (置信度: 0.134)
    3. bald eagle (置信度: 0.084)

正在识别: 004.jpg
  Top 3 预测结果:
    1. kite (置信度: 0.945)
    2. vulture (置信度: 0.017)
    3. bald eagle (置信度: 0.011)

正在识别: 005.jpg
  Top 3 预测结果:
    1. harvestman (置信度: 0.304)
    2. barn spider (置信度: 0.299)
    3. black widow (置信度: 0.294)

正在识别: af640d1b934568d3cd61a003444d0a2e.jpg
  Top 3 预测结果:
    1. king penguin (置信度: 0.812)
    2. American coot (置信度: 0.029)
    3. sloth bear (置信度: 0.020)

